In [0]:
/*
================================================================================
VISTA: evolucion_precios_carnes
CAPA: Gold (Analítica)
ENTORNO: DESARROLLO
PROPÓSITO: Proporcionar métricas de evolución temporal de precios de carnes
           normalizados para análisis de tendencias y variaciones
================================================================================

DESCRIPCIÓN:
Esta vista analítica permite monitorear la evolución diaria de los precios de 
productos cárnicos en diferentes supermercados, calculando métricas de cambio
absoluto y porcentual para identificar tendencias de mercado.

REGLAS DE NEGOCIO:

1. FILTRO DE CATEGORÍA:
   - Solo se incluyen productos de la categoría 'Carnes'
   - Filtro aplicado sobre la tabla silver ya limpia y normalizada

2. KPI: CAMBIO_DIARIO (valor absoluto)
   - Diferencia en pesos entre el precio actual y el precio del día anterior
   - Cálculo: precio_normalizado(t) - precio_normalizado(t-1)
   - Se particiona por producto_id para mantener continuidad del mismo producto
   - Se ordena por fecha_extraccion para mantener secuencia temporal
   - NULL cuando es el primer registro del producto (no hay día anterior)

3. KPI: CAMBIO_PORCENTUAL (variación %)
   - Variación porcentual del precio respecto al día anterior
   - Cálculo: [(precio(t) - precio(t-1)) / precio(t-1)] * 100
   - Redondeado a 2 decimales para facilitar interpretación
   - Permite comparar magnitud de cambios entre productos con diferentes rangos de precio
   - NULL cuando es el primer registro o cuando el precio anterior es 0

4. NORMALIZACIÓN DE PRECIOS:
   - Se utiliza precio_normalizado de la capa Silver
   - Garantiza que los precios sean comparables (misma unidad de medida)

5. IDENTIFICACIÓN TEMPORAL:
   - fecha_extraccion: determina el orden cronológico de los registros
   - LAG() respeta la ventana temporal por producto individual

USO RECOMENDADO:
- Dashboards de seguimiento de precios
- Alertas de variaciones significativas (ej: cambio_porcentual > 10%)
- Análisis de competitividad entre supermercados
- Detección de tendencias alcistas o bajistas en el mercado de carnes

NOTA: Esta es la versión de DESARROLLO. Para producción usar supermercadoetl_prod
================================================================================
*/

-- Paso 1: Crear schema Gold si no existe
CREATE SCHEMA IF NOT EXISTS supermercadoetl_dev.gold
COMMENT 'Capa Gold - Métricas para realizar Data Analytics';

-- Paso 2: Crear vista de evolución de precios de carnes
CREATE OR REPLACE VIEW supermercadoetl_dev.gold.evolucion_precios_carnes AS
SELECT 
  fecha_extraccion,
  nombre,
  supermercado,
  categoria,
  precio_normalizado,
  
  -- KPI 1: Cambio absoluto en pesos respecto al día anterior
  precio_normalizado - LAG(precio_normalizado) OVER (
    PARTITION BY producto_id 
    ORDER BY fecha_extraccion
  ) as cambio_diario,
  
  -- KPI 2: Cambio porcentual respecto al día anterior (redondeado a 2 decimales)
  ROUND(
    (precio_normalizado - LAG(precio_normalizado) OVER (
      PARTITION BY producto_id 
      ORDER BY fecha_extraccion
    )) 
    / LAG(precio_normalizado) OVER (
      PARTITION BY producto_id 
      ORDER BY fecha_extraccion
    ) * 100, 
    2
  ) as cambio_porcentual
  
FROM supermercadoetl_dev.silver.precios_productos
WHERE categoria = 'Carnes'
ORDER BY fecha_extraccion DESC, supermercado, nombre;